# Pair-prefix diagnostic — evaluation only, CE + CROSS, T4 ×2
Không training, optimizer, SCST hay cập nhật weights. GPU0 đọc CE; GPU1 đọc CROSS độc lập, không DDP collectives.
Chỉ kiểmtra 4 nhóm cùng ảnh: 2holdout đã xem +2fit controls. Không phải test độc lập.
Bắt buộc khôi phục đúng full-model hash và replay câu/score trước khi diễn giải can thiệp.
**Cần hai file adaptation.bin từ OUTPUT lần chạy CE và CROSS, không nằm trong review ZIP.**
Có thể đổi tên thành adaptation_ce.bin và adaptation_cross.bin khi tạo dataset. Đổi đuôi/đổi tên không đổi bytes.
Inputs: checkpoint nền gốc SHA5f8..., visual-cache, MSCOCOJSON+ảnh, hai adapter. Không dùng F.bin.
Chạy Save & Run All, InternetON, T4×2. Notebook này chưa được tác giả chạy model.


In [ ]:
print('PAIR PREFIX DIAGNOSTIC: START / NO TRAINING',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/qformer-scst-object-w05-legacy/qformer_scst_object_w05_legacy.pth')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
COCO_IMAGES=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/images')
# Sửa hai path nếu đã upload adapter thành dataset. KHÔNG trỏ vào checkpoint nền/F.
ADAPTER_CE=Path('/kaggle/working/phrase_decoder_ce_21/adaptation.bin')
ADAPTER_CROSS=Path('/kaggle/working/phrase_decoder_cross_21/adaptation.bin')
OUTPUT=Path('/kaggle/working/pair_prefix_diagnostic_4')


In [ ]:
import base64,hashlib,json,os,shutil,subprocess,sys,zlib
os.environ.update(PYTHONUNBUFFERED='1',HDF5_USE_FILE_LOCKING='FALSE',CUBLAS_WORKSPACE_CONFIG=':4096:8',PYTHONHASHSEED='42',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
REPO=Path('/tmp/Image_Captioning-pair-prefix');LOG=Path('/tmp/pair_prefix_boot.log');CONFIG=REPO/'prefix_config.json'
def run(args,cwd=None):
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    with LOG.open('a',encoding='utf-8') as log:
        process=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        code=process.wait()
    if code:raise subprocess.CalledProcessError(code,args)
for p in (CHECKPOINT,COCO_JSON,ADAPTER_CE,ADAPTER_CROSS):
    if not p.is_file():raise FileNotFoundError(f'Missing {p}. Set adapter paths to the two saved adaptation.bin files; review ZIPs do not contain weights.')
assert ADAPTER_CE.resolve()!=ADAPTER_CROSS.resolve(),'Need two different adapters; metadata/full hashes are checked below'
for p in (VISUAL_CACHE,COCO_IMAGES):assert p.is_dir(),p
assert not OUTPUT.exists(),'Use a fresh OUTPUT/session; no overwrite'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_name(i)=='Tesla T4' for i in (0,1)),'Select T4 x2'
if not REPO.exists():run(['git','clone','--no-checkout','https://github.com/Supzxjee/Image_Captioning.git',REPO])
run(['git','fetch','--depth','1','origin','d772e6854b55f88abd9d7fc45465181fc7d1726d'],REPO)
run(['git','checkout','--detach','d772e6854b55f88abd9d7fc45465181fc7d1726d'],REPO)
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow'])
PAYLOAD='...'
data=zlib.decompress(base64.b64decode(PAYLOAD,validate=True));assert hashlib.sha256(data).hexdigest()=='3b303c95a91c59cea67dce16a789953f87283b87d99132b88416650692e43821'
packet=json.loads(data);hashes={'phrase_data.py': '3d076b5257ad18c295d29607efb3cb81605d28075751073a40b3e3ae4d6e81be', 'phrase_model.py': 'cb3dba8f6e71698db9213f1dbd7137f8c5a0c3613d76765bcadb263ec4844834', 'run_phrase.py': 'd615ab32d64ff98d3d542124094f2cab8fedd59b3cb8fbb7d5ebc2bdf4c25997', 'phrase_review.py': 'd37b8b343776c6b6b60dc157ac5f11440ac769567424fcb3748134d0cf4a7353', 'test_phrase_data.py': 'ed136f70bb450d42256289d921577685e0589985989aabe222f7212e8b292ab7', 'test_phrase_model.py': 'a875e25eb7022108ac8105f709c6cd560fca90cd4b95787c798c1a020e6a611d', 'run_prefix_diagnostic.py': 'fba137af4a5e398f8d1fb311f745292ff1245fd11aca45dd444493f9481b14d3', 'test_prefix_data.py': '3b1892a2b9f6feaf4707bd6e25a83007a554479b55d1663806c55e7957275a1d'}
for name,source in packet['sources'].items():
    assert hashlib.sha256(source.encode()).hexdigest()==hashes[name]
    (REPO/name).write_text(source,encoding='utf-8')
manifest=REPO/'locked_manifest.json';manifest.write_text(packet['manifest'],encoding='utf-8')
cfg=dict(arm='CE',checkpoint=str(CHECKPOINT),visual_cache=str(VISUAL_CACHE),dataset_json=str(COCO_JSON),coco_images=str(COCO_IMAGES),
    output=str(OUTPUT),stage='/tmp/pair_prefix_stage.npz',manifest=str(manifest),manifest_sha256=hashlib.sha256(packet['manifest'].encode()).hexdigest(),
    source_sha256=hashes,project_commit='d772e6854b55f88abd9d7fc45465181fc7d1726d',adapters=dict(CE=str(ADAPTER_CE),CROSS=str(ADAPTER_CROSS)),expected=packet['expected'])
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8')
run([sys.executable,'-u','test_prefix_data.py'],REPO)
run([sys.executable,'-u','test_phrase_model.py'],REPO)


In [ ]:
# Frozen visual staging only: reuse the exact original preparation and tokenizer.
run([sys.executable,'-u','run_phrase.py','prepare','--config',CONFIG],REPO)
(OUTPUT/'source').mkdir()
for name in hashes:shutil.copy2(REPO/name,OUTPUT/'source'/name)
try:
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0','run_prefix_diagnostic.py','worker','--config',CONFIG],REPO)
    run([sys.executable,'-u','run_prefix_diagnostic.py','finish','--config',CONFIG],REPO)
finally:
    shutil.copy2(LOG,OUTPUT/'run.log')
    run([sys.executable,'-u','run_phrase.py','archive','--config',CONFIG],REPO)
print('SEND:',str(OUTPUT)+'_review.zip',flush=True)


## Diễn giải và gửi kết quả
Gửi pair_prefix_diagnostic_4_review.zip. Có top10 token, branch ranks/logP, logP từng token teacher-forced,
sinh từBOS / sau a person / sau prefixchung của hai mô tả; ảnh+box và log.
Natural: đổi sang cặp thật trong cùngảnh, G/P giốngnhau. Evidenceoff: chỉlà thamchiếu nhạy với nhánh.
Geometryswap: chỉtráo vector8D, giữ masks/membership; là canthiệp khôngnhấtquán/OOD, không chứngminh loại bỏ hếtgeometry.
Sau a person, branch_tokens của phone là holding/looking, nhưng prefix chung thực của hai câu phone chỉlà a person.
Nếu replay score/câu/hash thấtbại, dừng diễn giải và gửilog; không giảm tolerance hoặc bỏassert.
Không có semanticaccuracy tựđộng; không huấn luyện hoặc đề xuất routing từ riêng kếtquả này.
